In [1]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import Point
import geopandas as gpd
from pyproj import Transformer
import os

**chdir**

In [2]:
os.chdir('/home/bill/GitHub/wps-research/data/bill')

In [3]:
f = 'C11659/245/VNP14IMG.A2025245.1012.002.2025245191721.nc'

In [4]:
ds = nc.Dataset(f, 'r')

lat = ds['FP_latitude'][:]
lon = ds['FP_longitude'][:]
frp = ds['FP_power'][:]
conf = ds['FP_confidence'][:]

In [6]:
for attr in ds.ncattrs():
    print(f"{attr}: {ds.getncattr(attr)}")

ds.close()

VNP02IMG: VNP02CCIMG.A2025245.1012.002.2025245191056.nc
VNP03IMG: VNP03IMG.A2025245.1012.002.2025245184151.nc
VNP02MOD: VNP02CCMOD.A2025245.1012.002.2025245191056.nc
VNP02GDC: VNP02GDC.A2025245.1012.002.2025245190602.nc
ProcessVersionNumber: 3.1.9
ExecutableCreationDate: Oct 31 2023
ExecutableCreationTime: 17:46:00
SystemID: Linux minion20157 5.4.0-1119-fips #129-Ubuntu SMP Wed Apr 16 10:03:43 UTC 2025 x86_64
Unagg_GRingLatitude: 52.122581,59.574871,79.612236,65.609718
Unagg_GRingLongitude: -96.326035,-145.144257,-161.956512,-63.184685
NorthBoundingCoordinate: 80.8980484008789
SouthBoundingCoordinate: 52.122581481933594
EastBoundingCoordinate: -63.18468475341797
WestBoundingCoordinate: -161.95651245117188
Day/Night/Both: Both
FirePix: 711
DayPix: 5283466
LandPix: 4161515
ProcessingCenter: MODAPS-NASA
NightPix: 36290934
StartTime: 2025-09-02 10:12:00.000
WaterPix: 1121951
MissingPix: 12672
GlintPix: 0
CloudPix: 21750459
GRingPointLatitude: [52.1226 59.5749 79.6122 65.6097]
RangeEndingTi

In [11]:
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32609", always_xy=True)
utm_x, utm_y = transformer.transform(lon, lat)

In [12]:
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32609", always_xy=True)
utm_x, utm_y = transformer.transform(lon, lat)

In [13]:
df = pd.DataFrame({
    'latitude': lat,
    'longitude': lon,
    'utm_x': utm_x,
    'utm_y': utm_y,
    'FRP_MW': frp,
    'confidence': conf
})

In [14]:
geometry = [Point(xy) for xy in zip(utm_x, utm_y)]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:32609')

gdf.to_file('viirs_fires_utm9n.gpkg', driver='GPKG')